# Ward-42 Survey OCR Pipeline

**GPU Required:** Go to Settings → Accelerator → GPU T4 x2

This notebook runs the full SurveyPipeline on all 1446 PDFs:
1. Installs dependencies (PaddleOCR, MS-SWIFT, PyTorch CUDA)
2. Loads the OCR pipeline + Hindi VLM adapter
3. Processes every PDF → JSON + CSV + SQLite evidence store

**Required inputs:**
- `rk9098/ward42-surveys` — 1446 PDFs
- `rk9098/ocr-platform-code` — pipeline source code

**Outputs** (download from `/kaggle/working/output/` after run):
- `ocr_results.csv` — all extracted fields, flat
- `evidence.db` — full SQLite provenance store
- `documents/` — per-PDF JSON
- `run_manifest.json` — run metadata

Expected runtime: ~2–4 hours on T4

In [ ]:
%%time
#
# STEP 1: Install dependencies
# PaddleOCR requires specific paddlepaddle; install both carefully
#

!pip install -q --upgrade pip

# Core deps (CPU-safe)
!pip install -q \
    pypdfium2 \
    rapidfuzz \
    Pillow \
    pandas \
    numpy \
    opencv-contrib-python \
    pydantic \
    pyyaml

# PyTorch CUDA for T4 (CUDA 12.1)
!pip install -q \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

# PaddlePaddle GPU (pinned to avoid pipeline_1f1b import error)
!pip install -q "paddlepaddle-gpu==3.0.0.post119"

# PaddleOCR — use version compatible with paddlepaddle 3.0
!pip install -q "paddleocr>=3.0,<3.1"

# MS-SWIFT for Hindi VLM
!pip install -q ms-swift transformers accelerate sentencepiece

# Install the ocr-platform package from the code dataset
!pip install -q -e /kaggle/input/ocr-platform-code --no-deps

print("All dependencies installed.")
!nvidia-smi -L

In [ ]:
#
# STEP 1b: Patch PaddlePaddle pipeline_1f1b import bug
# PaddlePaddle 3.x has a known issue where pipeline_1f1b imports
# forward_complete_op_role from pass_utils but it's not exported.
# We add a no-op stub to make it import.
#
import os, sys
pp_path = os.path.join(sys.prefix, 'lib', 'python' + '.'.join(map(str, sys.version_info[:2])), 'site-packages')
pass_utils = os.path.join(pp_path, 'paddle', 'distributed', 'passes', 'pass_utils.py')

if os.path.exists(pass_utils):
    src = open(pass_utils).read()
    if 'def forward_complete_op_role' not in src:
        # Append the missing stub
        stub = '\n\ndef forward_complete_op_role(op):\n    return 0\n'
        with open(pass_utils, 'a') as f:
            f.write(stub)
        print(f'Patched {pass_utils}')
    else:
        print('Patch already applied')
else:
    print(f'pass_utils not found at {pass_utils}')

# Quick PaddleOCR smoke test
import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU available: {paddle.device.is_compiled_with_cuda()}')
from paddleocr import PaddleOCR
ocr_en = PaddleOCR(use_angle_cls=False, lang='en')
print('PaddleOCR English: OK')
try:
    ocr_hi = PaddleOCR(use_angle_cls=False, lang='hi')
    print('PaddleOCR Hindi (hi): OK')
except Exception as e:
    try:
        ocr_hi = PaddleOCR(use_angle_cls=False, lang='multilingual')
        print('PaddleOCR Hindi (multilingual): OK')
    except Exception as e2:
        print(f'PaddleOCR Hindi failed: {e} / {e2}')

In [ ]:
#
# STEP 1c: Download HindiOCR-VLM LoRA adapter
# The adapter is gated on HuggingFace — skip if not accessible.
# Pipeline falls back to PaddleOCR alone.
#
import os, subprocess, sys

HINDI_VLM_AVAILABLE = False
ADAPTER_DIR = '/kaggle/working/hindi_vlm_adapter'

result = subprocess.run(
    ['huggingface-cli', 'download', '--repo-type', 'model',
     '--local-dir', ADAPTER_DIR,
     '--local-dir-use-symlinks', 'False',
     '--quiet',
     'shaoncvit/HindiOCR-VLM'],
    capture_output=True, text=True, timeout=300
)

if result.returncode == 0 and (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    print(f'✓ HindiOCR-VLM adapter downloaded to {ADAPTER_DIR}')
    HINDI_VLM_AVAILABLE = True
else:
    print(f'⚠ HindiOCR-VLM adapter not available: {result.stderr[:200]}')
    print('  Pipeline will use PaddleOCR only (still effective for Hindi + English)')
    os.environ['USE_HINDI_VLM'] = '0'

# Set env var for pipeline
os.environ['HINDI_VLM_ADAPTER_DIR'] = ADAPTER_DIR if HINDI_VLM_AVAILABLE else ''

In [ ]:
#
# STEP 2: Run the full pipeline on all PDFs
#
import os, sys, json, time, csv
from pathlib import Path
from datetime import datetime

# --- Paths ---
PDF_ROOT  = Path('/kaggle/input/ward42-surveys')
WORK_ROOT = Path('/kaggle/working')
CODE_ROOT = Path('/kaggle/input/ocr-platform-code')
OUT_DIR   = WORK_ROOT / 'output'
DOCS_DIR  = OUT_DIR / 'documents'
DB_PATH   = OUT_DIR / 'evidence.db'
CSV_PATH  = OUT_DIR / 'ocr_results.csv'
MANIFEST  = OUT_DIR / 'run_manifest.json'

for d in [OUT_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE_ROOT / 'src'))

# --- Discover PDFs ---
pdfs = sorted(PDF_ROOT.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDFs in {PDF_ROOT}')

# --- Init pipeline ---
from ocr_platform.orchestrator.pipeline import SurveyPipeline

TEMPLATE = CODE_ROOT / 'configs' / 'ward42_template.json'

pipeline = SurveyPipeline(
    template_path=str(TEMPLATE),
    db_path=str(DB_PATH),
    output_dir=str(DOCS_DIR),
    dpi=150,
    use_hindi_vlm=HINDI_VLM_AVAILABLE,
)

print(f'Hindi VLM enabled: {HINDI_VLM_AVAILABLE}')
print(f'Template: {TEMPLATE}')
print(f'Output: {OUT_DIR}')

# --- Run ---
results  = []
errors   = []
t0 = time.time()

for i, pdf_path in enumerate(pdfs):
    doc_id = pdf_path.stem
    rel    = pdf_path.relative_to(PDF_ROOT)

    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(pdfs)}] {rel}")
    print(f"{'='*60}")

    try:
        result = pipeline.run(pdf_path)

        for member in result.get('members', []):
            row = {
                'doc_id':        doc_id,
                'sr_no':         member.get('sr_no', ''),
                'member_name':   member.get('member_name', ''),
                'relation':      member.get('relation', ''),
                'gender':        member.get('gender', ''),
                'age':           member.get('age', ''),
                'marital_status':member.get('marital_status', ''),
                'religion':      member.get('religion', ''),
                'caste':         member.get('caste', ''),
                'caste_category':member.get('caste_category', ''),
                'education':     member.get('education', ''),
                'occupation':    member.get('occupation', ''),
                'monthly_income':member.get('monthly_income', ''),
                'bank_account':  member.get('bank_account', ''),
                'resident_code': member.get('resident_code', ''),
                'migration_nature': member.get('migration_nature', ''),
                'remarks':       member.get('remarks', ''),
            }
            results.append(row)

        n_members = result.get('members_extracted', 0)
        n_anomalies = len(result.get('qa_anomalies', []))
        print(f'  ✓ {n_members} members | {n_anomalies} anomalies')

    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        import traceback; traceback.print_exc()
        errors.append({'doc_id': doc_id, 'error': str(e)})

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f'COMPLETE: {len(pdfs)-len(errors)}/{len(pdfs)} PDFs in {elapsed:.0f}s ({elapsed/60:.1f}min)')
print(f'Total members: {len(results)} | Errors: {len(errors)}')
print(f"{'='*60}")

# --- Write CSV ---
fieldnames = [
    'doc_id','sr_no','member_name','relation','gender','age',
    'marital_status','religion','caste','caste_category',
    'education','occupation','monthly_income','bank_account',
    'resident_code','migration_nature','remarks'
]
with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(results)

# --- Write manifest ---
manifest = {
    'run_id': f'kaggle_{datetime.utcnow():%Y%m%d_%H%M%S}',
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'total_pdfs': len(pdfs),
    'successful': len(pdfs) - len(errors),
    'errors': len(errors),
    'total_members': len(results),
    'elapsed_seconds': round(elapsed, 1),
    'hindi_vlm': HINDI_VLM_AVAILABLE,
    'error_list': errors,
}
with open(MANIFEST, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)

print(f'\n📄 CSV: {CSV_PATH}')
print(f'🗄️  DB: {DB_PATH}')
print(f'📁 Docs: {DOCS_DIR}')
print(f'📋 Manifest: {MANIFEST}')
print('\n✅ Done! Download /kaggle/working/output/ from the Output tab.')